In [ ]:
!pip install -U bitsandbytes datasets huggingface_hub transformers peft accelerate sentencepiece
# === Imports & GPU check ===
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [ ]:
import os
from torch.utils.data import TensorDataset, DataLoader

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))


torch: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4


In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"   # change if you want (LLaMA-3, etc.)
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128
TRAIN_EXAMPLES = 2000    # lower for quick tests
EVAL_EXAMPLES = 200
BATCH_SIZE = 1           # per device (use gradient_accumulation for effective larger batch)
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
EPOCHS = 1
OUTPUT_DIR = "./mistral_qloar_results"

# === Load dataset (we will avoid dataset.map) ===
dataset = load_dataset("ccdv/arxiv-summarization")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 203037
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6436
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6440
    })
})


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
# ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === Load model in 4-bit (QLoRA) using bitsandbytes ===
print("Loading model in 4-bit (QLoRA) ... this may take a moment")

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=False  # set True if model requires remote code; usually False is safer
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading model in 4-bit (QLoRA) ... this may take a moment


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
model = prepare_model_for_kbit_training(model)  # sets requires_grad appropriate, enables gradient checkpointing etc.

# === LoRA config and apply LoRA ===
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # good default for many transformer implementations
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # ch

trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


In [ ]:
def tokenize_examples(batch):
    # batch is a dict-style HuggingFace slice: e.g. dataset["train"][:N]
    inputs = ["Summarize this document:\n" + doc for doc in batch["article"]]
    tokenized = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=True,        # padding added here to create tensors of same length
        return_tensors="pt"
    )
    targets = tokenizer(
        batch["abstract"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=True, # padding added here
        return_tensors="pt"
    )
    # labels will be the token ids of targets; for causal LM Trainer expects labels aligned
    tokenized["labels"] = targets["input_ids"]
    return tokenized

In [ ]:
train_slice = dataset["train"][:TRAIN_EXAMPLES]
eval_slice = dataset["validation"][:EVAL_EXAMPLES]

train_tok = tokenize_examples(train_slice)
eval_tok = tokenize_examples(eval_slice)

# Build TensorDatasets of variable-length sequences is awkward; instead we store lists and write a collator
train_examples = list(zip(train_tok["input_ids"], train_tok["attention_mask"], train_tok["labels"]))
eval_examples = list(zip(eval_tok["input_ids"], eval_tok["attention_mask"], eval_tok["labels"]))

In [ ]:
from torch.utils.data import Dataset

class SimpleListDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        input_ids, attn_mask, labels = self.examples[idx]
        return {"input_ids": input_ids, "attention_mask": attn_mask, "labels": labels}

train_dataset = SimpleListDataset(train_examples)
eval_dataset = SimpleListDataset(eval_examples)

In [ ]:
def collate_fn(batch):
    # batch is a list of dicts with tensors of different lengths
    input_ids = [b["input_ids"] for b in batch]
    attn = [b["attention_mask"] for b in batch]
    labels = [b["labels"] for b in batch]

    # pad
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attn_padded = torch.nn.utils.rnn.pad_sequence(attn, batch_first=True, padding_value=0)

    # Pad labels to the same length as input_ids_padded
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)  # -100 ignored by loss
    if labels_padded.shape[1] < input_ids_padded.shape[1]:
        padding_length = input_ids_padded.shape[1] - labels_padded.shape[1]
        labels_padded = torch.nn.functional.pad(labels_padded, (0, padding_length), "constant", -100)
    elif labels_padded.shape[1] > input_ids_padded.shape[1]:
         # This case should ideally not happen with truncation, but added for robustness
         labels_padded = labels_padded[:, :input_ids_padded.shape[1]]


    print(f"Shape of padded input_ids: {input_ids_padded.shape}")
    print(f"Shape of padded labels: {labels_padded.shape}")


    return {
        "input_ids": input_ids_padded,
        "attention_mask": attn_padded,
        "labels": labels_padded
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",   # avoid eval if your transformers version complains; set to "epoch" if supported
    report_to="none",
    save_total_limit=2,
    dataloader_pin_memory=True,
)

# === Trainer ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
)

# === Train ===
trainer.train()

# === Save LoRA adapters (saves small weights) ===
peft_save_dir = os.path.join(OUTPUT_DIR, "lora-adapter")
model.save_pretrained(peft_save_dir)
print("Saved LoRA adapters to:", peft_save_dir)

# === Quick inference test (on CPU or GPU depending where model resides) ===
sample_text = dataset["test"][0]["article"]
prompt = "Summarize this document:\n" + sample_text[:2000]  # shorten to keep inference fast
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH)
# move inputs to same device as model's first param
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}
out = model.generate(**inputs, max_new_tokens=100)
print("SUMMARY:\n", tokenizer.decode(out[0], skip_special_tokens=True))

Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,6.537200
40,6.477400
60,6.451200
80,6.376500
100,6.419000
120,6.423500
140,6.315900
160,6.403500
180,6.541800
200,6.513100


Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels

Step,Training Loss
20,6.537200
40,6.477400
60,6.451200
80,6.376500
100,6.419000
120,6.423500
140,6.315900
160,6.403500
180,6.541800
200,6.513100


Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels: torch.Size([1, 512])
Shape of padded input_ids: torch.Size([1, 512])
Shape of padded labels

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Caching is incompatible with gradient checkpointing in MistralDecoderLayer. Setting `past_key_values=None`.


Saved LoRA adapters to: ./mistral_qloar_results/lora-adapter


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


SUMMARY:
 Summarize this document:
for about 20 years the problem of properties of short - term changes of solar activity has been considered extensively . 
 many investigators studied the short - term periodicities of the various indices of solar activity . 
 several periodicities were detected , but the periodicities about 155 days and from the interval of @xmath3 $ ] days ( @xmath4 $ ] years ) are mentioned most often . 
 first of them was discovered by @xcite in the occurence rate of gamma - ray flares detected by the gamma - ray spectrometer aboard the _ solar maximum mission ( smm ) . 
 this periodicity was confirmed for other solar flares data and for the same time period @xcite . 
 it was also found in proton flares during solar cycles 19 and 20 @xcite , but it was not found in the solar flares data during solar cycles 22 @xcite . 
 _    several autors confirmed above results for the daily sunspot area data . @xcite studied the sunspot data from 18741984 . 
 she found the 155-d